# The EM Algorithm

**Companion lesson:** https://ml-viz.vercel.app/courses/probabilistic-models/02-em-algorithm

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'

## EM for 1D GMM

E-step: compute responsibilities $\gamma_{ik} = P(z_i = k | x_i)$

M-step: update $\mu_k, \sigma_k, \pi_k$ using soft assignments

In [ ]:
np.random.seed(42)
n = 300
X = np.concatenate([np.random.randn(150) * 0.8 - 2, np.random.randn(150) * 1.2 + 3])

# Initialize
K = 2
mu = np.array([-3.0, 2.0])
sigma = np.array([1.0, 1.0])
pi = np.array([0.5, 0.5])

log_likelihoods = []
snapshots = []

for iteration in range(30):
    # E-step
    gamma = np.zeros((len(X), K))
    for k in range(K):
        gamma[:, k] = pi[k] * norm.pdf(X, mu[k], sigma[k])
    gamma /= gamma.sum(axis=1, keepdims=True)
    
    # Log-likelihood
    ll = np.sum(np.log(sum(pi[k] * norm.pdf(X, mu[k], sigma[k]) for k in range(K))))
    log_likelihoods.append(ll)
    
    if iteration in [0, 1, 5, 15, 29]:
        snapshots.append((iteration, mu.copy(), sigma.copy(), pi.copy()))
    
    # M-step
    Nk = gamma.sum(axis=0)
    for k in range(K):
        mu[k] = np.sum(gamma[:, k] * X) / Nk[k]
        sigma[k] = np.sqrt(np.sum(gamma[:, k] * (X - mu[k])**2) / Nk[k])
        pi[k] = Nk[k] / len(X)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

x_grid = np.linspace(-6, 8, 300)
for i, (iter_num, mu_s, sigma_s, pi_s) in enumerate(snapshots):
    ax = axes[i]
    ax.hist(X, bins=40, density=True, color='#818cf8', alpha=0.3, edgecolor='#1a1d27')
    mixture = sum(pi_s[k] * norm.pdf(x_grid, mu_s[k], sigma_s[k]) for k in range(K))
    ax.plot(x_grid, mixture, color='#14b8a6', linewidth=2)
    for k in range(K):
        ax.plot(x_grid, pi_s[k] * norm.pdf(x_grid, mu_s[k], sigma_s[k]), '--',
                color=['#f43f5e', '#eab308'][k], linewidth=1)
    ax.set_title(f'Iteration {iter_num}', color='white', fontsize=11)
    ax.set_xlim(-6, 8)

# Log-likelihood plot
ax = axes[5]
ax.plot(log_likelihoods, color='#818cf8', linewidth=2)
ax.set_xlabel('Iteration')
ax.set_ylabel('Log-Likelihood')
ax.set_title('Monotonic Increase (EM guarantee)', color='white', fontsize=11)

plt.suptitle('EM Algorithm: Iterative Convergence', color='white', fontsize=13)
plt.tight_layout()
plt.show()

## Model Selection: BIC vs AIC

How many components should we use? Both penalize complexity.

In [ ]:
from sklearn.mixture import GaussianMixture

bic_scores = []
aic_scores = []
for k in range(1, 8):
    gmm = GaussianMixture(n_components=k, random_state=42).fit(X.reshape(-1, 1))
    bic_scores.append(gmm.bic(X.reshape(-1, 1)))
    aic_scores.append(gmm.aic(X.reshape(-1, 1)))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, 8), bic_scores, 'o-', color='#818cf8', linewidth=2, label='BIC')
ax.plot(range(1, 8), aic_scores, 's-', color='#14b8a6', linewidth=2, label='AIC')
ax.axvline(2, color='#f43f5e', linestyle='--', alpha=0.5, label='True K = 2')
ax.set_xlabel('Number of Components')
ax.set_ylabel('Score (lower is better)')
ax.set_title('BIC vs AIC for Model Selection', color='white')
ax.legend()
plt.tight_layout()
plt.show()

## The log-likelihood increases every iteration

EM is guaranteed to **never decrease** the log-likelihood — it climbs to a local optimum. Watching the curve is the standard convergence check.

In [ ]:
from sklearn.mixture import GaussianMixture
from sklearn.datasets import make_blobs

X, _ = make_blobs(n_samples=400, centers=3, cluster_std=1.0, random_state=0)
lls = []
for n in range(1, 21):
    gm = GaussianMixture(n_components=3, max_iter=n, n_init=1,
                         init_params='random', random_state=2).fit(X)
    lls.append(gm.lower_bound_)         # avg log-likelihood at convergence of n iters
plt.plot(range(1, 21), lls, 'o-', color='#14b8a6')
plt.xlabel('max EM iterations'); plt.ylabel('avg log-likelihood')
plt.title('EM monotonically improves the likelihood'); plt.show()

## Key takeaways

- **EM** alternates an **E-step** (soft responsibilities) and **M-step** (re-estimate parameters).
- It handles **latent variables** — which Gaussian generated each point is unknown.
- The log-likelihood is **non-decreasing**; EM converges to a **local** optimum (use restarts).
- The same E/M pattern powers HMMs (Baum-Welch), missing-data imputation, and more.